In [7]:
# ------------------------------------------------------------
# STEP 1: LOAD DATA
# ------------------------------------------------------------

import pandas as pd
import numpy as np

url = "gs://agntworks-data-dev/sandbox/experiments/clean_flight_data_with_regime_v3.csv"

df = pd.read_csv(url, parse_dates=["dep_datetime"])

print("Shape:", df.shape)
df.head()

Shape: (10794, 12)


,Trip_Number,Trip_Legs_ID,dep_date,dep_datetime,Dep_Time_Actual_GMT,Dep_cluster,Arr_cluster,Quote_Total_Cost,Revenue_allocated,Block_Hours,RevHr_clean,regime
0,157565,42366991KODR3217,2016-01-01,2016-01-01 23:37:00,23:37:00,WASHINGTON_DC_CLUSTER,OTHER_CLUSTER,44251.20,11954.115640,2.850000,4194.426540,NORMAL
1,157565,420331372FOSJ3517,2016-01-02,2016-01-02 18:00:00,18:00:00,OTHER_CLUSTER,MIAMI_CLUSTER,44251.20,6291.639810,1.500000,4194.426540,PEAK_MIXED
2,157565,420331372FOSJ4217,2016-01-02,2016-01-02 20:42:00,20:42:00,MIAMI_CLUSTER,SAN_FRANCISCO_CLUSTER,44251.20,26005.444550,6.200000,4194.426540,PEAK_MIXED
3,162215,42125996CASG317,2016-01-10,2016-01-10 14:56:00,14:56:00,BOSTON_CLUSTER,OTHER_CLUSTER,76864.32,22011.546392,4.166667,5282.771134,HIGH_YIELD
4,162215,423761209HARN017,2016-01-10,2016-01-10 21:10:00,21:10:00,OTHER_CLUSTER,SAN_JUAN_CLUSTER,76864.32,6251.279175,1.183333,5282.771134,HIGH_YIELD


In [8]:
# Create the 'corridor' feature by combining departure and arrival clusters
df['corridor'] = df['Dep_cluster'] + ' - ' + df['Arr_cluster']

# Display the first few rows to verify the new column
df[['Dep_cluster', 'Arr_cluster', 'corridor']].head()

,Dep_cluster,Arr_cluster,corridor
0,WASHINGTON_DC_CLUSTER,OTHER_CLUSTER,WASHINGTON_DC_CLUSTER - OTHER_CLUSTER
1,OTHER_CLUSTER,MIAMI_CLUSTER,OTHER_CLUSTER - MIAMI_CLUSTER
2,MIAMI_CLUSTER,SAN_FRANCISCO_CLUSTER,MIAMI_CLUSTER - SAN_FRANCISCO_CLUSTER
3,BOSTON_CLUSTER,OTHER_CLUSTER,BOSTON_CLUSTER - OTHER_CLUSTER
4,OTHER_CLUSTER,SAN_JUAN_CLUSTER,OTHER_CLUSTER - SAN_JUAN_CLUSTER


In [9]:
df.head()

,Trip_Number,Trip_Legs_ID,dep_date,dep_datetime,Dep_Time_Actual_GMT,Dep_cluster,Arr_cluster,Quote_Total_Cost,Revenue_allocated,Block_Hours,RevHr_clean,regime,corridor
0,157565,42366991KODR3217,2016-01-01,2016-01-01 23:37:00,23:37:00,WASHINGTON_DC_CLUSTER,OTHER_CLUSTER,44251.20,11954.115640,2.850000,4194.426540,NORMAL,WASHINGTON_DC_CLUSTER - OTHER_CLUSTER
1,157565,420331372FOSJ3517,2016-01-02,2016-01-02 18:00:00,18:00:00,OTHER_CLUSTER,MIAMI_CLUSTER,44251.20,6291.639810,1.500000,4194.426540,PEAK_MIXED,OTHER_CLUSTER - MIAMI_CLUSTER
2,157565,420331372FOSJ4217,2016-01-02,2016-01-02 20:42:00,20:42:00,MIAMI_CLUSTER,SAN_FRANCISCO_CLUSTER,44251.20,26005.444550,6.200000,4194.426540,PEAK_MIXED,MIAMI_CLUSTER - SAN_FRANCISCO_CLUSTER
3,162215,42125996CASG317,2016-01-10,2016-01-10 14:56:00,14:56:00,BOSTON_CLUSTER,OTHER_CLUSTER,76864.32,22011.546392,4.166667,5282.771134,HIGH_YIELD,BOSTON_CLUSTER - OTHER_CLUSTER
4,162215,423761209HARN017,2016-01-10,2016-01-10 21:10:00,21:10:00,OTHER_CLUSTER,SAN_JUAN_CLUSTER,76864.32,6251.279175,1.183333,5282.771134,HIGH_YIELD,OTHER_CLUSTER - SAN_JUAN_CLUSTER


In [10]:
# Shows the count of missing values per column
df.isnull().sum()

Trip_Number            0
Trip_Legs_ID           0
dep_date               0
dep_datetime           0
Dep_Time_Actual_GMT    0
Dep_cluster            0
Arr_cluster            0
Quote_Total_Cost       0
Revenue_allocated      0
Block_Hours            0
RevHr_clean            0
regime                 0
corridor               0
dtype: int64

In [11]:
# Count how many flights are in each corridor
corridor_counts = df['corridor'].value_counts().reset_index()
corridor_counts.columns = ['corridor', 'flight_count']

# Show the top 10 most frequent corridors
corridor_counts.head(10)

,corridor,flight_count
0,MIAMI_CLUSTER - NEW_YORK_CLUSTER,293
1,NEW_YORK_CLUSTER - MIAMI_CLUSTER,241
2,SAN_FRANCISCO_CLUSTER - SAN_FRANCISCO_CLUSTER,215
3,NEW_YORK_CLUSTER - NEW_YORK_CLUSTER,190
4,BOSTON_CLUSTER - NEW_YORK_CLUSTER,178
5,LOS_ANGELES_CLUSTER - SAN_FRANCISCO_CLUSTER,170
6,NEW_YORK_CLUSTER - BOSTON_CLUSTER,169
7,SAN_FRANCISCO_CLUSTER - LOS_ANGELES_CLUSTER,165
8,BOSTON_CLUSTER - BOSTON_CLUSTER,144
9,MIAMI_CLUSTER - MIAMI_CLUSTER,136


In [12]:
# Group by corridor to see total and average revenue impact
impact_analysis = df.groupby('corridor').agg({
    'Trip_Number': 'count',           # Volume (Number of flights)
    'Revenue_allocated': 'sum',       # Total Revenue Impact
    'RevHr_clean': 'mean'             # Efficiency (Avg Revenue per Hour)
}).rename(columns={'Trip_Number': 'Flight_Count'}).sort_values(by='Flight_Count', ascending=False)

# Display the top corridors
impact_analysis.head(15)

,Flight_Count,Revenue_allocated,RevHr_clean
corridor,,,
MIAMI_CLUSTER - NEW_YORK_CLUSTER,293,4.539167e+06,6334.819669
NEW_YORK_CLUSTER - MIAMI_CLUSTER,241,3.732947e+06,6098.603766
SAN_FRANCISCO_CLUSTER - SAN_FRANCISCO_CLUSTER,215,6.406267e+05,8155.906824
NEW_YORK_CLUSTER - NEW_YORK_CLUSTER,190,5.669942e+05,8432.787610
BOSTON_CLUSTER - NEW_YORK_CLUSTER,178,1.130057e+06,7570.188329
LOS_ANGELES_CLUSTER - SAN_FRANCISCO_CLUSTER,170,1.364936e+06,8453.430106
NEW_YORK_CLUSTER - BOSTON_CLUSTER,169,1.037140e+06,8341.759800
SAN_FRANCISCO_CLUSTER - LOS_ANGELES_CLUSTER,165,1.230083e+06,8373.202222
BOSTON_CLUSTER - BOSTON_CLUSTER,144,6.322113e+05,7714.708661


In [13]:
# 1. Identify which corridors have 80 or more flights
counts = df['corridor'].value_counts()
major_corridors = counts[counts >= 80].index

# 2. Create a new dataframe containing only those major corridors
df_high_impact = df[df['corridor'].isin(major_corridors)].copy()

# 3. Verify the result
print(f"Total rows in original data: {len(df)}")
print(f"Rows in high-impact data (>= 80 flights): {len(df_high_impact)}")
print(f"Number of unique 'Major' corridors: {df_high_impact['corridor'].nunique()}")

df_high_impact.head()

Total rows in original data: 10794
Rows in high-impact data (>= 80 flights): 3233
Number of unique 'Major' corridors: 23


,Trip_Number,Trip_Legs_ID,dep_date,dep_datetime,Dep_Time_Actual_GMT,Dep_cluster,Arr_cluster,Quote_Total_Cost,Revenue_allocated,Block_Hours,RevHr_clean,regime,corridor
8,162216,423961225KODR8017,2016-02-01,2016-02-01 12:40:00,12:40:00,NEW_YORK_CLUSTER,BOSTON_CLUSTER,80640.00,4564.528302,0.550000,8299.142367,NORMAL,NEW_YORK_CLUSTER - BOSTON_CLUSTER
24,167652,423651387KODR517,2016-01-02,2016-01-02 19:35:00,19:35:00,NEW_YORK_CLUSTER,CHICAGO_CLUSTER,48705.24,23929.096174,1.883333,12705.714783,PEAK_MIXED,NEW_YORK_CLUSTER - CHICAGO_CLUSTER
27,167832,42366994KODR117,2016-01-02,2016-01-02 12:15:00,12:15:00,NEW_YORK_CLUSTER,OTHER_CLUSTER,35275.20,18101.747368,3.250000,5569.768421,PEAK_MIXED,NEW_YORK_CLUSTER - OTHER_CLUSTER
28,167832,422421314MACJ317,2016-01-02,2016-01-02 16:58:00,16:58:00,OTHER_CLUSTER,NEW_YORK_CLUSTER,35275.20,17173.452632,3.083333,5569.768421,PEAK_MIXED,OTHER_CLUSTER - NEW_YORK_CLUSTER
45,168392,423661343KODR4117,2016-01-03,2016-01-03 12:45:00,12:45:00,NEW_YORK_CLUSTER,NEW_YORK_CLUSTER,21371.19,1401.389508,0.200000,7006.947541,PEAK_MIXED,NEW_YORK_CLUSTER - NEW_YORK_CLUSTER


In [14]:
# Group by corridor to see efficiency on major routes
high_impact_summary = df_high_impact.groupby('corridor').agg({
    'Revenue_allocated': 'mean',    # Average money per trip
    'RevHr_clean': 'mean',          # Average money per HOUR (Efficiency)
    'Block_Hours': 'mean'           # Average flight duration
}).sort_values(by='RevHr_clean', ascending=False)

high_impact_summary.head(10)

,Revenue_allocated,RevHr_clean,Block_Hours
corridor,,,
LOS_ANGELES_CLUSTER - LOS_ANGELES_CLUSTER,2948.683212,8594.412315,0.748899
LOS_ANGELES_CLUSTER - SAN_FRANCISCO_CLUSTER,8029.034878,8453.430106,1.461569
NEW_YORK_CLUSTER - NEW_YORK_CLUSTER,2984.180162,8432.787610,0.541667
SAN_FRANCISCO_CLUSTER - LOS_ANGELES_CLUSTER,7455.050427,8373.202222,1.276667
NEW_YORK_CLUSTER - BOSTON_CLUSTER,6136.925920,8341.759800,0.973373
SAN_FRANCISCO_CLUSTER - SAN_FRANCISCO_CLUSTER,2979.659094,8155.906824,0.483411
BOSTON_CLUSTER - BOSTON_CLUSTER,4390.356054,7714.708661,0.703935
NEW_YORK_CLUSTER - WASHINGTON_DC_CLUSTER,6650.721975,7572.431986,0.884035
BOSTON_CLUSTER - NEW_YORK_CLUSTER,6348.636490,7570.188329,0.813015


In [21]:
# Check for price consistency
consistency_check = df_high_impact.groupby('corridor')['Revenue_allocated'].std().sort_values(ascending=False)
print("Corridors with the most inconsistent pricing:")
print(consistency_check.head(5))

Corridors with the most inconsistent pricing:
corridor
OTHER_CLUSTER - NEW_YORK_CLUSTER            12264.100034
NEW_YORK_CLUSTER - OTHER_CLUSTER            12061.642095
NEW_YORK_CLUSTER - SAN_FRANCISCO_CLUSTER     8489.616592
MIAMI_CLUSTER - BOSTON_CLUSTER               8215.603057
SAN_FRANCISCO_CLUSTER - NEW_YORK_CLUSTER     7859.412546
Name: Revenue_allocated, dtype: float64


In [22]:
df_high_impact.to_csv('high_impact_corridors.csv', index=False)

In [24]:
# ------------------------------------------------------------
# Gemini setup
# ------------------------------------------------------------
# If needed:
# !pip install -q google-genai
# ------------------------------------------------------------

from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project="agntworks-dev", location="us-central1")

In [28]:
response = client.models.generate_content(
    model="gemini-2.5-pro",
    contents=[
        types.Content(role="user", parts=[types.Part(text="say hi" + "\n\n" + "print hello world")])
    ]
)

print(response.text)

Hi!
hello world
